In [1]:
import os

import numpy as np
import pandas as pd

In [2]:
RATINGS_PATH = None
COURSES_PATH = None
OUT_DIR = "outputs"
SEED = 42

N_FACTORS = 8
EPOCHS = 80
LEARNING_RATE = 0.02
REGULARIZATION = 0.2
TOP_N = 6

LEVEL_ORDER = {"beginner": 0, "intermediate": 1, "advanced": 2}

os.makedirs(OUT_DIR, exist_ok=True)

In [3]:
def make_sample_data(n_interns=80, courses_per_intern=14):
    rng = np.random.RandomState(SEED)
    categories = ["Python", "Data Analysis", "Machine Learning", "Web Development", "Communication"]
    levels = ["Beginner", "Intermediate", "Advanced"]
    suffix = {"Beginner": "Fundamentals", "Intermediate": "in Practice", "Advanced": "Advanced Topics"}

    courses = []
    course_id = 1
    for cat in categories:
        for level in levels:
            for part in ("I", "II"):
                courses.append({
                    "course_id": course_id,
                    "title": f"{cat} {suffix[level]} {part}",
                    "category": cat,
                    "level": level,
                    "duration_hours": int(rng.choice([4, 6, 8, 10, 12])),
                })
                course_id += 1
    courses = pd.DataFrame(courses)

    rows = []
    for intern_id in range(1, n_interns + 1):
        favored = rng.choice(categories, size=2, replace=False)
        intern_level = rng.randint(0, 3)
        picked = rng.choice(courses.index, size=courses_per_intern, replace=False)
        for idx in picked:
            course = courses.loc[idx]
            base = 4.3 if course["category"] in favored else 2.6
            if LEVEL_ORDER[course["level"].lower()] > intern_level:
                base -= 0.5
            rating = int(np.clip(round(base + rng.normal(0, 0.6)), 1, 5))
            rows.append({"intern_id": intern_id, "course_id": int(course["course_id"]), "rating": rating})
    return pd.DataFrame(rows), courses

In [12]:
if RATINGS_PATH is None or COURSES_PATH is None:
    ratings, courses = make_sample_data()
else:
    ratings = pd.read_csv(RATINGS_PATH)
    courses = pd.read_csv(COURSES_PATH)

for col in ("intern_id", "course_id", "rating"):
    if col not in ratings.columns:
        raise ValueError(f"ratings file must contain '{col}'")
for col in ("course_id", "title"):
    if col not in courses.columns:
        raise ValueError(f"courses file must contain '{col}'")

ratings = ratings.dropna(subset=["intern_id", "course_id", "rating"])
ratings = ratings.drop_duplicates(subset=["intern_id", "course_id"], keep="last")
ratings = ratings[ratings["course_id"].isin(courses["course_id"])].reset_index(drop=True)

for col, default in (("category", "General"), ("level", "Intermediate"), ("duration_hours", 0)):
    if col not in courses.columns:
        courses[col] = default

courses = courses.drop_duplicates(subset="course_id").reset_index(drop=True)
courses["level_rank"] = courses["level"].astype(str).str.lower().map(LEVEL_ORDER).fillna(1).astype(int)

intern_ids = np.sort(ratings["intern_id"].unique())
course_ids = courses["course_id"].to_numpy()
intern_index = {iid: n for n, iid in enumerate(intern_ids)}
course_index = {cid: n for n, cid in enumerate(course_ids)}

data = np.column_stack([
    ratings["intern_id"].map(intern_index).to_numpy(),
    ratings["course_id"].map(course_index).to_numpy(),
    ratings["rating"].to_numpy(dtype=float),
])

n_users, n_items = len(intern_ids), len(course_ids)
density = len(ratings) / (n_users * n_items)
print(f"Interns: {n_users} | Courses: {n_items} | Ratings: {len(ratings)} | Matrix density: {density:.1%}")
ratings["rating"].value_counts().sort_index()

Interns: 80 | Courses: 30 | Ratings: 1120 | Matrix density: 46.7%


,count
rating,
1,52
2,303
3,345
4,290
5,130


In [5]:
def train_mf(train, n_users, n_items, k, epochs, lr, reg, seed):
    rng = np.random.RandomState(seed)
    P = rng.normal(0, 0.1, (n_users, k))
    Q = rng.normal(0, 0.1, (n_items, k))
    bu = np.zeros(n_users)
    bi = np.zeros(n_items)
    mu = train[:, 2].mean()
    order = np.arange(len(train))

    for _ in range(epochs):
        rng.shuffle(order)
        for row in order:
            u, i, r = int(train[row, 0]), int(train[row, 1]), train[row, 2]
            err = r - (mu + bu[u] + bi[i] + P[u] @ Q[i])
            bu[u] += lr * (err - reg * bu[u])
            bi[i] += lr * (err - reg * bi[i])
            p_old = P[u].copy()
            P[u] += lr * (err * Q[i] - reg * P[u])
            Q[i] += lr * (err * p_old - reg * Q[i])

    return {"mu": mu, "bu": bu, "bi": bi, "P": P, "Q": Q}


def score_matrix(model):
    scores = model["mu"] + model["bu"][:, None] + model["bi"][None, :] + model["P"] @ model["Q"].T
    return np.clip(scores, 1, 5)


def rmse(scores, test):
    users, items = test[:, 0].astype(int), test[:, 1].astype(int)
    return float(np.sqrt(np.mean((scores[users, items] - test[:, 2]) ** 2)))


def ranking_metrics(scores, train, test, k=5, threshold=4):
    seen, relevant = {}, {}
    for u, i, r in train:
        seen.setdefault(int(u), set()).add(int(i))
    for u, i, r in test:
        if r >= threshold:
            relevant.setdefault(int(u), set()).add(int(i))

    hits, precisions = [], []
    for u, rel in relevant.items():
        s = scores[u].copy()
        for i in seen.get(u, ()):
            s[i] = -np.inf
        top = set(np.argsort(-s)[:k].tolist())
        n_hit = len(rel & top)
        hits.append(n_hit > 0)
        precisions.append(n_hit / k)
    return float(np.mean(hits)), float(np.mean(precisions))

In [6]:
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(data))
cut = int(len(data) * 0.8)
train, test = data[perm[:cut]], data[perm[cut:]]

eval_model = train_mf(train, n_users, n_items, N_FACTORS, EPOCHS, LEARNING_RATE, REGULARIZATION, SEED)
mf_scores = score_matrix(eval_model)

mean_scores = np.full((n_users, n_items), train[:, 2].mean())
popularity = np.bincount(train[:, 1].astype(int), minlength=n_items).astype(float)
pop_scores = np.tile(popularity, (n_users, 1))

mf_hit, mf_prec = ranking_metrics(mf_scores, train, test)
pop_hit, pop_prec = ranking_metrics(pop_scores, train, test)

pd.DataFrame({
    "RMSE": [rmse(mf_scores, test), rmse(mean_scores, test)],
    "Hit rate @5": [mf_hit, pop_hit],
    "Precision @5": [mf_prec, pop_prec],
}, index=["Matrix Factorization", "Baseline (global mean / popularity)"]).round(3)

,RMSE,Hit rate @5,Precision @5
Matrix Factorization,0.951,0.655,0.145
Baseline (global mean / popularity),1.043,0.397,0.083


In [13]:
final_model = train_mf(data, n_users, n_items, N_FACTORS, EPOCHS, LEARNING_RATE, REGULARIZATION, SEED)
final_scores = score_matrix(final_model)
final_popularity = np.bincount(data[:, 1].astype(int), minlength=n_items).astype(float)

completed = {}
for u, i, r in data:
    completed.setdefault(int(u), set()).add(int(i))


def recommend_path(intern_id, top_n=TOP_N):
    if intern_id in intern_index:
        u = intern_index[intern_id]
        scores = final_scores[u].copy()
        for i in completed.get(u, ()):
            scores[i] = -np.inf
        label = "predicted_rating"
    else:
        scores = final_popularity.copy()
        label = "popularity"

    top = np.argsort(-scores)[:top_n]
    path = courses.iloc[top][["course_id", "title", "category", "level", "duration_hours", "level_rank"]].copy()
    path[label] = np.round(scores[top], 2)
    path = path.sort_values(["level_rank", label], ascending=[True, False]).drop(columns="level_rank")
    path.insert(0, "step", range(1, len(path) + 1))
    return path.reset_index(drop=True)

## Example recommendations

In [8]:
demo_intern = int(intern_ids[0])
history = ratings[ratings["intern_id"] == demo_intern].merge(courses[["course_id", "title", "category"]], on="course_id")
print(f"Intern {demo_intern}: highest-rated past courses")
history.sort_values("rating", ascending=False).head(5)[["title", "category", "rating"]]

Intern 1: highest-rated past courses


,title,category,rating
0,Data Analysis Advanced Topics I,Data Analysis,4
1,Data Analysis in Practice II,Data Analysis,4
11,Data Analysis Fundamentals II,Data Analysis,4
4,Python Advanced Topics II,Python,3
7,Web Development Fundamentals II,Web Development,3


In [9]:
recommend_path(demo_intern)

,step,course_id,title,category,level,duration_hours,predicted_rating
0,1,7,Data Analysis Fundamentals I,Data Analysis,Beginner,8,3.35
1,2,2,Python Fundamentals II,Python,Beginner,12,2.96
2,3,9,Data Analysis in Practice I,Data Analysis,Intermediate,8,3.26
3,4,15,Machine Learning in Practice I,Machine Learning,Intermediate,10,2.87
4,5,12,Data Analysis Advanced Topics II,Data Analysis,Advanced,8,3.28
5,6,5,Python Advanced Topics I,Python,Advanced,12,2.90


In [10]:
recommend_path(int(intern_ids[1]))

,step,course_id,title,category,level,duration_hours,predicted_rating
0,1,1,Python Fundamentals I,Python,Beginner,10,3.77
1,2,2,Python Fundamentals II,Python,Beginner,12,3.58
2,3,10,Data Analysis in Practice II,Data Analysis,Intermediate,12,3.43
3,4,24,Web Development Advanced Topics II,Web Development,Advanced,4,4.15
4,5,23,Web Development Advanced Topics I,Web Development,Advanced,10,3.86
5,6,5,Python Advanced Topics I,Python,Advanced,12,3.81


New interns with no history receive the most popular courses (cold start).

In [11]:
recommend_path(-1)

,step,course_id,title,category,level,duration_hours,popularity
0,1,7,Data Analysis Fundamentals I,Data Analysis,Beginner,8,45.0
1,2,14,Machine Learning Fundamentals II,Machine Learning,Beginner,6,43.0
2,3,16,Machine Learning in Practice II,Machine Learning,Intermediate,6,45.0
3,4,10,Data Analysis in Practice II,Data Analysis,Intermediate,12,43.0
4,5,9,Data Analysis in Practice I,Data Analysis,Intermediate,8,40.0
5,6,6,Python Advanced Topics II,Python,Advanced,6,45.0
